In [117]:
print("hello world!!")

hello world!!


In [118]:
#setup
%matplotlib inline
from __future__ import print_function
from six.moves import zip, range
import pandas as pd
import numpy as np
# import jellyfish
# from collections import OrderedDict
import splink as sp
from splink import DuckDBAPI, Linker, SettingsCreator, block_on, splink_datasets
db_api=DuckDBAPI()

In [119]:
# read in sq land transactions (known)
df_complete_lt = pd.read_stata('/Users/maggiewang/Downloads/Year_4/Thesis/ncomplete_clean.dta')

In [120]:
# browse data
df_complete_lt.head()

,province,lnarea,land_grade,lnprice,date,unique_id
0,山东省,3.1892,二级,8.531096,2008-09-01,1.0
1,北京市,0.8929,七级,4.897350,2007-04-01,2.0
2,北京市,0.1339,七级,2.489073,2007-06-01,3.0
3,北京市,0.0436,三级,6.230633,2007-05-01,4.0
4,北京市,1.6770,一级,9.896825,2007-05-01,5.0


In [121]:
# read in princelings land transactions (censored)
df_princelings_lt = pd.read_stata('/Users/maggiewang/Downloads/Year_4/Thesis/dataverse_files/princelings_nclean.dta')

In [122]:
# browse data
df_princelings_lt.head()

,lnarea,land_grade,province,lnprice,unique_id,date
0,2.201659,15.0,1.0,7.438384,1.0,2013-07-01
1,5.995730,19.0,1.0,6.696465,2.0,2014-03-01
2,2.201659,15.0,1.0,7.438384,3.0,2013-07-01
3,5.893659,13.0,1.0,4.231656,4.0,2009-02-01
4,6.903109,13.0,1.0,6.396930,5.0,2014-08-01


In [123]:
province1 = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]
province2 = ["Shanghai", "Tianjin", "Hunan", "Fujian", "Anhui", "Yunnan", "Inner Mongolia", "Jiangxi", "Ningxia", "Zhejiang", "Jiangsu", "Hubei", "Jilin", "Heilongjiang", "Guangdong", "Sichuan", "Shandong", "Hebei", "Henan", "Guizhou", "Hainan", "Gansu", "Guangxi", "Shanxi", "Liaoning", "Tibet", "Chongqing", "Shaanxi", "Beijing", "Xinjiang", "Xinjiang", "Qinghai"]

df_princelings_lt['province'] = df_princelings_lt['province'].replace(province1, province2)

In [124]:
df_princelings_lt['province'].value_counts()

province
Zhejiang          104098
Guangdong         103215
Jiangsu            86120
Hunan              64686
Shandong           63755
Hubei              62979
Yunnan             61055
Sichuan            57102
Guangxi            54090
Hebei              53319
Henan              52107
Anhui              45503
Xinjiang           42747
Heilongjiang       40327
Liaoning           36583
Inner Mongolia     34531
Jiangxi            34460
Jilin              33508
Guizhou            27349
Fujian             25018
Chongqing          21954
Shanxi             20109
Shaanxi            18617
Gansu              17921
Tianjin            12361
Shanghai            9058
Beijing             7376
Ningxia             7106
Qinghai             3938
Hainan              3037
Tibet                 31
Name: count, dtype: int64

In [125]:
province1 = ["浙江省", "广东省", "江苏省", "湖南省", "山东省", "湖北省", "云南省", "四川省", "广西壮族", "河北省", "河南省", "安徽省", "黑龙江省", "辽宁省", "内蒙古", "江西省", "吉林省", "新疆维吾尔", "贵州省", "福建省", "重庆市", "山西省", "陕西省", "甘肃省", "天津市", "新疆建设兵团", "上海市", "北京市", "宁夏回族", "青海省", "海南省", "西藏"]

province2 = [
    "Zhejiang",        # 浙江省
    "Guangdong",       # 广东省
    "Jiangsu",         # 江苏省
    "Hunan",           # 湖南省
    "Shandong",        # 山东省
    "Hubei",           # 湖北省
    "Yunnan",          # 云南省
    "Sichuan",         # 四川省
    "Guangxi",         # 广西壮族
    "Hebei",           # 河北省
    "Henan",           # 河南省
    "Anhui",           # 安徽省
    "Heilongjiang",    # 黑龙江省
    "Liaoning",        # 辽宁省
    "Inner Mongolia",  # 内蒙古
    "Jiangxi",         # 江西省  
    "Jilin",           # 吉林省
    "Xinjiang",        # 新疆维吾尔
    "Guizhou",         # 贵州省
    "Fujian",          # 福建省
    "Chongqing",       # 重庆市
    "Shanxi",          # 山西省
    "Shaanxi",         # 陕西省
    "Gansu",           # 甘肃省
    "Tianjin",         # 天津市
    "Xinjiang",        # 新疆建设兵团
    "Shanghai",        # 上海市
    "Beijing",         # 北京市
    "Ningxia",         # 宁夏回族
    "Qinghai",         # 青海省
    "Hainan",          # 海南省
    "Tibet"            # 西藏
]

df_complete_lt['province'] = df_complete_lt['province'].replace(province1, province2)

##check
df_complete_lt['province'].value_counts()

province
Hunan             91734
Shandong          87996
Guangdong         85939
Hubei             81928
Sichuan           76622
Jiangsu           73417
Zhejiang          72204
Yunnan            70955
Guangxi           62399
Hebei             48606
Anhui             46454
Xinjiang          45050
Henan             42647
Inner Mongolia    39891
Liaoning          36778
Jiangxi           32738
Jilin             28593
Fujian            25272
Guizhou           25230
Heilongjiang      24731
Shanxi            19188
Chongqing         19129
Shaanxi           18737
Gansu             16500
Tianjin           11118
Ningxia            6823
Qinghai            5877
Shanghai           5150
Beijing            5023
Hainan             3919
Tibet                24
Name: count, dtype: int64

In [126]:
df_princelings_lt['land_grade'].value_counts()

land_grade
19.0    188716
17.0    177052
1.0     149426
16.0    139419
18.0    122927
15.0     81353
7.0      50335
6.0      44656
8.0      40017
14.0     37691
9.0      35069
11.0     34975
13.0     31213
10.0     29857
12.0     24344
5.0      18932
2.0       1721
4.0        581
3.0        337
Name: count, dtype: int64

In [127]:
type(df_complete_lt['land_grade'])

pandas.core.series.Series

In [128]:
##cleaning complete dataset land_grade values, leaving only graded/ungraded values
dfcompletegraded = df_complete_lt.loc[
    df_complete_lt['land_grade'].str.contains("级|区", na=False)
]

##check
dfcompletegraded['land_grade'].value_counts()

land_grade
一级       225264
三级       183126
二级       149656
未评估地区    130861
四级       123345
五级        62965
十三等级      54574
十四等级      48736
十二级       40770
十一级       30059
六级        29519
九级        28561
十级        25955
十五等级      25653
七级        23633
八级        21000
十八等级       1941
十六等级        893
十七等级        378
土地级别          1
Name: count, dtype: int64

In [129]:
##cleaning complete dataset, leaving out outlier observation
dfcgraded = dfcompletegraded[~dfcompletegraded['land_grade'].str.contains("土地", na=False)]

##check
dfcgraded['land_grade'].value_counts()

land_grade
一级       225264
三级       183126
二级       149656
未评估地区    130861
四级       123345
五级        62965
十三等级      54574
十四等级      48736
十二级       40770
十一级       30059
六级        29519
九级        28561
十级        25955
十五等级      25653
七级        23633
八级        21000
十八等级       1941
十六等级        893
十七等级        378
Name: count, dtype: int64

In [130]:
## recoding land_grade values in princelings dataset
grade1 = [19.0, 17.0, 1.0, 16.0, 18.0, 15.0, 7.0, 6.0, 8.0, 14.0, 9.0, 11.0, 13.0, 10.0, 12.0, 5.0, 2.0, 4.0, 3.0]
grade2 = [1, 3, 0, 4, 2, 5, 13, 14, 12, 6, 11, 9, 7, 10, 8, 15, 18, 16, 17]

df_princelings_lt['land_grade'] = df_princelings_lt['land_grade'].replace(grade1, grade2)

#check for accuracy
df_princelings_lt['land_grade'].value_counts()

land_grade
1.0     188716
3.0     177052
0.0     149426
4.0     139419
2.0     122927
5.0      81353
13.0     50335
14.0     44656
12.0     40017
6.0      37691
11.0     35069
9.0      34975
7.0      31213
10.0     29857
8.0      24344
15.0     18932
18.0      1721
16.0       581
17.0       337
Name: count, dtype: int64

In [131]:
## recoding land_grade values in complete dataset
grade1 = ["一级", "未评估地区", "三级", "二级", "四级", "五级", "十三等级", "十四等级", "十二级", "六级", "十一级", "十五等级", "九级", "十级", "七级", "八级", "十八等级", "十六等级", "十七等级"]
grade2 = [1, 0, 3, 2, 4, 5, 13, 14, 12, 6, 11, 15, 9, 10, 7, 8, 18, 16, 17]

pd.set_option('future.no_silent_downcasting', True)
df_complete_lt['land_grade'] = dfcgraded['land_grade'].replace(grade1, grade2).astype(float)

#check for accuracy
df_complete_lt['land_grade'].value_counts()

land_grade
1.0     225264
3.0     183126
2.0     149656
0.0     130861
4.0     123345
5.0      62965
13.0     54574
14.0     48736
12.0     40770
11.0     30059
6.0      29519
9.0      28561
10.0     25955
15.0     25653
7.0      23633
8.0      21000
18.0      1941
16.0       893
17.0       378
Name: count, dtype: int64

In [132]:
df_complete_lt.to_csv('/Users/maggiewang/Downloads/Year_4/Thesis/coded_complete.csv')
df_princelings_lt.to_csv('/Users/maggiewang/Downloads/Year_4/Thesis/dataverse_files/coded_princelings.csv')

In [133]:
#browse
df_complete_lt.head()

,province,lnarea,land_grade,lnprice,date,unique_id
0,Shandong,3.1892,2.0,8.531096,2008-09-01,1.0
1,Beijing,0.8929,7.0,4.897350,2007-04-01,2.0
2,Beijing,0.1339,7.0,2.489073,2007-06-01,3.0
3,Beijing,0.0436,3.0,6.230633,2007-05-01,4.0
4,Beijing,1.6770,1.0,9.896825,2007-05-01,5.0


In [134]:
df_princelings_lt.head()

,lnarea,land_grade,province,lnprice,unique_id,date
0,2.201659,5.0,Shanghai,7.438384,1.0,2013-07-01
1,5.995730,1.0,Shanghai,6.696465,2.0,2014-03-01
2,2.201659,5.0,Shanghai,7.438384,3.0,2013-07-01
3,5.893659,7.0,Shanghai,4.231656,4.0,2009-02-01
4,6.903109,7.0,Shanghai,6.396930,5.0,2014-08-01
